# Call ROH from WGS data and bam input
Uses more than the 1240k SNPs - the input now are all SNPs in 1000G with >5%MAF. That allows for ROH calls in aDNA WGS data down to ~0.1x coverage

In [1]:
### First some Standard Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os as os
import sys as sys
import multiprocessing as mp
print(f"CPU Count: {mp.cpu_count()}")

### If you want to use a version other than the default installed hapROH 
### uncomment the following and set the path to the correct (installed) package
# sys.path.insert(0,"/mnt/archgen/users/pflorence/packages/hapROH/package/")  # Uncomment to get local package first in path

CPU Count: 128


# 1) Prepare the input data
First, we need to generate an eigenstrat file with the 1000G SNPs.

We begin by defining the input and output paths - you can adjust them to your use case. The input .bam should be a processed one, e.g. the one that one would use to call pseudo-haploid genotypes. This is usually the last stage of your processing pipeline.

The function below grabs the SNP set from the MAF 5% reference panel (path_h5) and produces a matching eigenstrat genotype file from your .bam.

## [Download test data]
To run this test, you can download the necessary example data and reference panel:
- WGS example .bam file: Use [this Dropbox folder](https://www.dropbox.com/scl/fo/9wqnhka1clwduaf375oyy/AOXi2b-3BshPXTeoaCTjMOQ?rlkey=ipc3b53ryjreruta8r6d40a4q&st=v9u8360n&dl=0). This data is ~0.15x Coverage, below the cutoff of classical hapROH with 1240k SNPs!
- Reference data for >5% MAF WGS data [this Dropbox folder](https://www.dropbox.com/scl/fo/i0s02rgh7m76fnhakkqg4/AEZLpJGvU6R6Tll6DtW9vp8?rlkey=c6ribb29mq46sgkcmdw7eojwu&st=y4hcd0du&dl=0)

In [ ]:
# 1000G reference panel with MAF 5% (you can download this from ...)
prefix_refHDF5 = "/mnt/archgen/users/yilei/Data/1000G/1000g1240khdf5/all1240/maf5_auto/maf5_chr"   

### Specific Target
path_bam = "/mnt/archgen/users/hringbauer/git/hapROH/Notebooks/Vignettes/ExampleData/wgs/SB606.merged.rg.markdup.indrealn_recalibrated.bam" # Update to the example data!
sample_name = "SB606" # The iid of your sample (to use in hapROH output)

prefix_outHDF5 = f"./ExampleData/med_jews/hdf5_maf5_1kg/" # path to the output directory
folder_roh_out = './ExampleData/med_jews/roh/'

### 1a) Prepare hdf5 file

In [3]:
from hapROH.IO.prep_input import bam2hdf5s

In [4]:
%%time
bam2hdf5s(path_bam, prefix_refHDF5, prefix_outHDF5, sample_name, overwrite=True)

Extracting SNP positions from hdf5
	Kept 530434/530434 biallelic SNP sites
Running mpileup on BAM file
	 63580/78921 positions covered, at mean depth 1.1123623781063228
	 Ignored 282/63580 positions containing indels
	 Found 55/70341=0.0782% bases different from expected ref/alt
	 Ignored 48/63298 positions with bases different from expected ref/alt
Extracting SNP positions from hdf5
	Kept 566080/566080 biallelic SNP sites
Running mpileup on BAM file
	 68123/84013 positions covered, at mean depth 1.1128546893119797
	 Ignored 278/68123 positions containing indels
	 Found 39/75457=0.0517% bases different from expected ref/alt
	 Ignored 39/67845 positions with bases different from expected ref/alt
Extracting SNP positions from hdf5
	Kept 490291/490291 biallelic SNP sites
Running mpileup on BAM file
	 59976/73229 positions covered, at mean depth 1.1131285847672403
	 Ignored 230/59976 positions containing indels
	 Found 35/66451=0.0527% bases different from expected ref/alt
	 Ignored 34/597

# 2) Run hapROH with WGS ref panel
Now that the eigenstrat file is prepared, you can run hapROH on it using the 1000G reference panel (with 5% MAF cutoff).

In [5]:
from hapsburg.PackagesSupport.hapsburg_run import hapsb_ind
from hapsburg.figures.plot_posterior import plot_posterior_cm

In [6]:
%%time
hapsb_ind(iid=sample_name, chs=range(1,23), 
           path_targets_prefix=prefix_outHDF5, # The directory contains files of format $iid.chr$chr.hdf5
           h5_path1000g=prefix_refHDF5, # The path up to the chr. number
           meta_path_ref="/mnt/archgen/users/yilei/Data/1000G/1000g1240khdf5/all1240/meta_df_all.csv", 
           folder_out=folder_roh_out,  # Folder where you want to save the results to 
           processes=4, output=True,
           p_model="HDF5",
           readcounts=True, logfile=True, combine=True)

Doing Individual SB606...
Set Output Log path: .ExampleData/med_jews/roh/SB606/chr3/hmm_run_log.txtSet Output Log path: .ExampleData/med_jews/roh/SB606/chr5/hmm_run_log.txtSet Output Log path: .ExampleData/med_jews/roh/SB606/chr7/hmm_run_log.txt

Set Output Log path: .ExampleData/med_jews/roh/SB606/chr1/hmm_run_log.txt

CPU times: user 17.8 ms, sys: 71.7 ms, total: 89.6 ms
Wall time: 162 ms


KeyError: "Unable to synchronously open object (object 'GT' doesn't exist)"

# 3) Make a plot of the inferred ROH

In [ ]:
from hapsburg.figures.plot_individual_roh import plot_roh_individual

In [ ]:
plot_roh_individual(iid=sample_name, folder="/mnt/archgen/users/hringbauer/data/med_jews/roh/", figsize=(6.5, 7),
                    prefix_out="", min_cm=4, plot_bad=False, savepath="") 

So we successfully found multiple ROH! This plot is as expected, as from this indivdual also a higher coverage version exists, which shows the same ROH. Congratulations on running hapROH for WGS samples!